## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 3 · Day 14–15 — Analysis Wrap-Up & Poster Figures  🏁 **Checkpoint 3**

You've done real neuroscience data science. Now we **consolidate**: gather your
best results into a tidy summary and a few poster-ready figures. By the end you'll
have everything you need to start building your poster in Week 4.

This notebook is mostly **yours to drive** — it's a guided checklist, not a
tutorial. Use code and ideas from any earlier notebook.

### Checkpoint 3 goal
A short results summary + 2–3 final figures saved in `outputs/`, plus a 5-minute
story you can tell about what you found.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
features = pd.read_csv(cu.save_path("features_table.csv"))

## 1. Pull together what you've saved
Each earlier notebook saved a CSV in `outputs/`. Let's see what you have.

In [ ]:
import os
out = cu.OUTPUT_DIR
if out.exists():
    print("Files in your outputs/ folder:")
    for f in sorted(os.listdir(out)):
        print("  ", f)
else:
    print("No outputs yet — run the earlier notebooks first.")

## 2. Your headline numbers
A poster needs a few crisp, true statements. Let's recompute the key ones so
they're fresh, then you'll write them in plain English.

In [ ]:
# load the effect-size table from Notebook 08 (if you ran it)
try:
    eff = pd.read_csv(cu.save_path("effect_sizes.csv"))
    top = eff.reindex(eff["hedges_g"].abs().sort_values(ascending=False).index).iloc[0]
    print(f"Strongest single feature: {top['feature']}  (Hedges' g = {top['hedges_g']:+.2f})")
except FileNotFoundError:
    print("Run Notebook 08 to get effect sizes.")

# load classifier result from Notebook 13/14 (if you ran it)
try:
    preds = pd.read_csv(cu.save_path("loocv_predictions.csv"))
    from sklearn.metrics import roc_auc_score
    print(f"Classifier AUC: {roc_auc_score(preds['true'], preds['prob']):.3f}")
except FileNotFoundError:
    print("Run Notebook 13 to get a classifier AUC.")

### ✏️ Your turn #1 — write your three findings
In plain English, fill in three true sentences about YOUR results. Be specific and
honest (mention n=28 and "exploratory" where appropriate).

In [ ]:
finding_1 = "..."   # TODO: e.g. "Sound-sensitive brains showed stronger gamma..."
finding_2 = "..."   # TODO
finding_3 = "..."   # TODO

for i, f in enumerate([finding_1, finding_2, finding_3], 1):
    print(f"Finding {i}: {f}")

cu.check(all(f != "..." and len(f) > 20 for f in [finding_1, finding_2, finding_3]),
         "Three findings written. These are your poster's backbone.",
         "Write a real sentence (20+ chars) for each finding.")

## 3. A polished summary figure
Here's a clean "money figure" template: your single best feature shown as a
box+strip comparison with the effect size in the title. Swap in *your* best
feature.

In [ ]:
import seaborn as sns
sns.set_theme(style="ticks", context="talk")   # bigger fonts for a poster

best_feature = "let_gamma"   # TODO: set to YOUR strongest feature

exp = features.loc[features["group"] == "EXP", best_feature].dropna()
ctrl = features.loc[features["group"] == "CTRL", best_feature].dropna()
g = cu.hedges_g(exp.values, ctrl.values)

fig, ax = plt.subplots(figsize=(6, 6))
palette = {"EXP": cu.EXP_COLOR, "CTRL": cu.CTRL_COLOR}
sns.boxplot(data=features, x="group", y=best_feature,
            palette=palette, width=0.5, fliersize=0, ax=ax)
sns.stripplot(data=features, x="group", y=best_feature,
              color="black", size=6, alpha=0.6, jitter=0.15, ax=ax)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel(best_feature)
ax.set_title(f"{best_feature}\nHedges' g = {g:+.2f}")
sns.despine()
plt.tight_layout()
plt.savefig(cu.save_path("poster_money_figure.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved poster_money_figure.png")

## 4. Assemble a 2-panel poster figure
### ✏️ Your turn #2 — combine two of your figures' *data* into one image
Build a single figure with **two panels** that together tell your story. For
example: (a) your best group comparison, (b) your spatial map OR your ROC curve
OR your loudness curve. Label them **a** and **b**, save at `dpi=300`.

*Hint:* `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))`, then draw into
`ax1` and `ax2` using code patterns from earlier notebooks.

In [ ]:
# TODO: your 2-panel poster figure here
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
# ... panel a into ax1 ...
# ... panel b into ax2 ...
# for ax, letter in zip((ax1, ax2), "ab"):
#     ax.set_title(letter, loc="left", fontweight="bold")
# plt.savefig(cu.save_path("poster_two_panel.png"), dpi=300, bbox_inches="tight")
# plt.show()

## 5. The story checklist
A great poster answers these in order. Jot a one-line answer for each (as
comments) — this becomes your 5-minute talk:

1. **Why care?** (What is hyperacusis? Why no objective test today?)
2. **What did we measure?** (ear-EEG, 4 tasks, 28 people)
3. **What did you find?** (your 3 findings)
4. **How confident are you?** (effect sizes, FDR, permutation, small n)
5. **What's next?** (more subjects? best task for screening?)

In [ ]:
# YOUR STORY (one line each):
# 1. Why care:
# 2. What measured:
# 3. What found:
# 4. How confident:
# 5. What's next:

## 🏁 Checkpoint 3 — you're ready for Week 4
- [ ] `outputs/` has your key figures saved at 300 DPI
- [ ] Three honest, specific findings written down
- [ ] A money figure + a two-panel figure for the poster
- [ ] A 5-point story you can tell out loud

🎉 **Congratulations** — you analyzed real clinical brain data end to end:
loading, features, statistics, effect sizes, correlations, and (for Tier 3)
machine learning with honest validation. Next week you turn it into a poster and
present your discovery. Go tell your story!